## **1. Carga de Datos y Preprocesamiento**

### 📊 Dataset: Red Social de Meetup (Tennessee)

Este conjunto de datos ofrece una visión detallada de las interacciones entre usuarios y grupos en **meetup.com**, una plataforma diseñada para organizar y asistir a eventos presenciales. Es un recurso ideal para aplicar **análisis de grafos** y teoría de redes.

---

### 📝 Contexto

Las relaciones entre quién asiste a qué evento forman una red social compleja. Este dataset fue creado originalmente para la charla *"Principles of Network Analysis with NetworkX"*, presentada en conferencias como **PyNash** y **PyTennessee**.

A través de estos datos, se exploran los fundamentos de la teoría de grafos utilizando **NetworkX** (una biblioteca de Python) para extraer información sobre el tejido social de los grupos de Meetup en Tennessee.

---

### 📂 Contenido del Dataset

La información está dividida en dos categorías principales: datos de grafos (aristas) y metadatos (nodos).

#### 🕸️ Datos de Grafos (Aristas)

Estos archivos contienen las conexiones y los "pesos" que definen la fuerza de cada relación.

| Archivo | Descripción | Peso (Weight) |
| --- | --- | --- |
| `member-to-group-edges.csv` | Red bipartita entre miembros y grupos. | Número de eventos asistidos. |
| `group-edges.csv` | Conexiones entre grupos. | Miembros compartidos entre grupos. |
| `member-edges.csv` | Conexiones entre miembros. | Grupos compartidos entre personas. |
| `rsvps.csv` | Datos crudos de asistencia. | Base para generar la red de miembros y grupos. |

### ℹ️ Metadatos

Información descriptiva para enriquecer el análisis de los nodos.

* **`meta-groups.csv`**: Detalles de cada grupo (nombre, categoría). Usa `group_id` como índice.
* **`meta-members.csv`**: Detalles de los usuarios (nombre, ubicación). Usa `member_id` como índice.
* **`meta-events.csv`**: Detalles de los eventos (nombre, fecha/hora). Usa `event_id` como índice.

In [4]:
import os
import kagglehub
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### **1.1. Ingesta desde Kaggle y Almacenamiento Dinámico**

El primer paso es descargar el dataset de la red de Meetup de Nashville directamente desde Kaggle (`stkbailey/nashville-meetup`). 

Para facilitar la reproducibilidad de este proyecto y no depender siempre de la descarga online, iteraremos sobre los ficheros bajados para copiarlos directamente a nuestra carpeta local `data/raw/`. 

A la vez, subimos los ficheros a memoria en un **diccionario de DataFrames** llamado `dataframes`, donde la clave será el nombre original del archivo (ej. `meta-members`).

*Nota: Durante la carga a memoria eliminamos la columna superficial `Unnamed: 0`, un residuo habitual al exportar CSVs desde Pandas que solo contiene el índice.*

In [69]:
import os
import shutil
import pandas as pd
import kagglehub

# Aseguramos que la carpeta data/raw existe
raw_dir = "../data/raw"
os.makedirs(raw_dir, exist_ok=True)

# Define las variables de la descarga
path_kaggle = kagglehub.dataset_download("stkbailey/nashville-meetup")
archivos = os.listdir(path_kaggle)

# Creamos el diccionario para guardar los DataFrames
dataframes = {}

print("--- Cargando Archivos y copiando a RAW ---")

for archivo in archivos:
    # 1. Copiar el CSV original a la carpeta local data/raw/
    src_file = os.path.join(path_kaggle, archivo)
    dst_file = os.path.join(raw_dir, archivo)
    shutil.copy2(src_file, dst_file)
    
    # 2. Cargar en Pandas
    df = pd.read_csv(src_file)
    
    if 'Unnamed: 0' in df.columns:
        df.drop('Unnamed: 0', axis=1, inplace=True)
         
    nombre_clave = archivo.replace(".csv", "")
    dataframes[nombre_clave] = df
                
    filas = df.shape[0]
    columnas = df.shape[1]
    
    # Mostramos que se ha cargado y guardado en local
    print(f"✅ Cargado y guardado: {nombre_clave} ({filas} filas x {columnas} columnas)")
    print("-" * 50)

print("--- Ingesta Completa ---")
print(f"Diccionario de DataFrames creado con claves:\n{list(dataframes.keys())}")


--- Cargando Archivos y copiando a RAW ---
✅ Cargado y guardado: group-edges (6692 filas x 3 columnas)
--------------------------------------------------
✅ Cargado y guardado: member-edges (1176368 filas x 3 columnas)
--------------------------------------------------
✅ Cargado y guardado: member-to-group-edges (45583 filas x 3 columnas)
--------------------------------------------------
✅ Cargado y guardado: meta-events (19307 filas x 4 columnas)
--------------------------------------------------
✅ Cargado y guardado: meta-groups (602 filas x 7 columnas)
--------------------------------------------------
✅ Cargado y guardado: meta-members (24591 filas x 7 columnas)
--------------------------------------------------
✅ Cargado y guardado: rsvps (126813 filas x 3 columnas)
--------------------------------------------------
--- Ingesta Completa ---
Diccionario de DataFrames creado con claves:
['group-edges', 'member-edges', 'member-to-group-edges', 'meta-events', 'meta-groups', 'meta-memb

### **1.2. Exploración Rápida de Dimensiones, Atributos y Tipos de Datos**

Antes de entrar en el preprocesamiento exhaustivo, es fundamental conocer la **estructura subyacente** de cada DataFrame.

Iteramos sobre el diccionario para imprimir las **columnas disponibles** en cada tabla. Esto nos ayuda a identificar rápidamente:
- Qué columnas actúan como identificadores de nodo (`member_id`, `group_id`, `event_id`).
- Qué características (features) tenemos disponibles para enriquecer los nodos en el futuro (`lat`, `lon`, `category_id`).
- Cuáles representan las aristas y sus pesos en los diferentes grafos (`weight`).

Al imprimir las columnas y los **tipos de datos (`dtypes`)**, buscamos:
- **Identificadores y claves foráneas**: (`member_id`, `group_id`, `event_id`) para cruzar tablas.
- **Features numéricas continuas**: Coordenadas (`lat`, `lon`), necesarias para cálculos espaciales si los hubiera.
- **Features categóricas/texto**: Nombres, ciudades o categorías (`category_name`), que podrían requerir un *encoding* a futuro.

In [70]:
for df in dataframes.keys():
    print(df)
    print(dataframes[df].columns)
    print("-" * 20)

group-edges
Index(['group1', 'group2', 'weight'], dtype='object')
--------------------
member-edges
Index(['member1', 'member2', 'weight'], dtype='object')
--------------------
member-to-group-edges
Index(['member_id', 'group_id', 'weight'], dtype='object')
--------------------
meta-events
Index(['event_id', 'group_id', 'name', 'time'], dtype='object')
--------------------
meta-groups
Index(['group_id', 'group_name', 'num_members', 'category_id', 'category_name',
       'organizer_id', 'group_urlname'],
      dtype='object')
--------------------
meta-members
Index(['member_id', 'name', 'hometown', 'city', 'state', 'lat', 'lon'], dtype='object')
--------------------
rsvps
Index(['event_id', 'member_id', 'group_id'], dtype='object')
--------------------


In [71]:
for df in dataframes.keys():
    print(dataframes[df].dtypes)

group1    int64
group2    int64
weight    int64
dtype: object
member1    int64
member2    int64
weight     int64
dtype: object
member_id    int64
group_id     int64
weight       int64
dtype: object
event_id    object
group_id     int64
name        object
time        object
dtype: object
group_id          int64
group_name       object
num_members       int64
category_id       int64
category_name    object
organizer_id      int64
group_urlname    object
dtype: object
member_id      int64
name          object
hometown      object
city          object
state         object
lat          float64
lon          float64
dtype: object
event_id     object
member_id     int64
group_id      int64
dtype: object


### **1.3 Calidad de Datos (Data Quality)**

Antes de alimentar cualquier red neuronal de grafos o algoritmo de detección de anomalías, debemos garantizar la **integridad matemática** de las tablas base. Este proceso se divide en dos revisiones fundamentales que deben ejecutarse en este estricto orden:

#### 1.3.1 Detección de Duplicados
Buscamos filas idénticamente repetidas en nuestros DataFrames.
* **Por qué es el primer paso:** Siempre eliminamos los duplicados antes de tratar los nulos. Las filas clonadas inflan artificialmente las estadísticas (ej. contando un mismo nulo múltiples veces) y pueden distorsionar el **peso (weight)** de las aristas en el grafo de forma involuntaria.


In [72]:
for name, df in dataframes.items():
    total_duplicates = df.duplicated().sum()
    print(f"{name} - Total de filas duplicadas: {total_duplicates}")
    print("-" * 40)

group-edges - Total de filas duplicadas: 0
----------------------------------------
member-edges - Total de filas duplicadas: 0
----------------------------------------
member-to-group-edges - Total de filas duplicadas: 0
----------------------------------------
meta-events - Total de filas duplicadas: 0
----------------------------------------
meta-groups - Total de filas duplicadas: 0
----------------------------------------
meta-members - Total de filas duplicadas: 0
----------------------------------------
rsvps - Total de filas duplicadas: 0
----------------------------------------


#### 1.3.2 Detección de Valores Nulos (NaN)
Una vez asegurada la unicidad de las filas, contabilizamos las variables faltantes.
* **Impacto en Grafos:** Un nodo con metadatos nulos puede quedar lógicamente aislado o inducir *bias* durante el entrenamiento (como en el GAE). Aquí se toma la decisión de eliminar filas corruptas o imputar valores mediante estrategias matemáticas (media, mediana o la etiqueta "Desconocido").

In [73]:
for name, df in dataframes.items():
    total_nulls = df.isna().sum().sum()  # Suma primero por columna y luego todo junto
    print(f"{name} - Total de valores nulos: {total_nulls}")
    print("-" * 40)

group-edges - Total de valores nulos: 0
----------------------------------------
member-edges - Total de valores nulos: 0
----------------------------------------
member-to-group-edges - Total de valores nulos: 0
----------------------------------------
meta-events - Total de valores nulos: 0
----------------------------------------
meta-groups - Total de valores nulos: 0
----------------------------------------
meta-members - Total de valores nulos: 19758
----------------------------------------
rsvps - Total de valores nulos: 0
----------------------------------------


In [74]:
print(dataframes["meta-members"].isna().sum())
print("-" * 40)
print((dataframes["meta-members"].isna().sum() / len(dataframes["meta-members"]) * 100).round(2).astype(str) + '%')

member_id        0
name             0
hometown     19664
city             0
state           94
lat              0
lon              0
dtype: int64
----------------------------------------
member_id      0.0%
name           0.0%
hometown     79.96%
city           0.0%
state         0.38%
lat            0.0%
lon            0.0%
dtype: object


In [75]:
dataframes["meta-members"] = dataframes["meta-members"].drop(columns="hometown", axis=1)

Al analizar la **`meta-members`**, observamos la siguiente distribución de valores nulos:
- La columna **`hometown`** tiene **19.664 valores nulos**, lo que representa un masivo **79.96%** de los datos faltantes.
- La columna **`state`** apenas presenta un **0.38%** de nulos.

Dado que casi el 80% de la información en `hometown` (ciudad natal) no existe, cualquier intento de imputación (ej. rellenarla con la moda general) introduciría un sesgo inaceptable en nuestra red que desvirtuaría los cálculos de similitud entre nodos posteriormente. 
**Decisión:** Eliminar la columna `hometown` completa.

En cuanto a "estado", como es menos de un 1%, se podría borrar simplemente esa docena de filas concretas o usar técnicas de geolocalización inversa usando las coordenadas `lat`/`lon` que sí están completas al 100% (lo analizaremos en `02_eda.ipynb`).

### **1.4 Exportación de Datos Limpios**

Una vez validada la calidad de los datos (sin nulos críticos ni duplicados que distorsionen los posteriores análisis topológicos), el último paso es consolidar este trabajo guardando los DataFrames resultantes.

Estos ficheros limpios se almacenan en el directorio `data/processed/`. Al guardarlos sin índices auto-generados (`index=False`), garantizamos que los identificadores originales (`group_id`, `member_id`, etc.) mantengan su estructura. 

A partir de este punto, **los notebooks de Análisis Exploratorio (EDA) y Modelado se alimentarán exclusivamente de este repositorio procesado**, asegurando la reproducibilidad y agilizando la carga de datos en futuras iteraciones computacionales.

In [76]:
# Asegurarse de que la carpeta existe
out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

for name, df in dataframes.items():
    # Guarda el fichero, sin añadir la columna de los índices de pandas
    df.to_csv(f"{out_dir}/{name}.csv", index=False)
    
    filas = df.shape[0]
    columnas = df.shape[1]
    print(f"✅ Guardado: {name}.csv ({filas} filas x {columnas} columnas)")
    print("-" * 50)

✅ Guardado: group-edges.csv (6692 filas x 3 columnas)
--------------------------------------------------
✅ Guardado: member-edges.csv (1176368 filas x 3 columnas)
--------------------------------------------------
✅ Guardado: member-to-group-edges.csv (45583 filas x 3 columnas)
--------------------------------------------------
✅ Guardado: meta-events.csv (19307 filas x 4 columnas)
--------------------------------------------------
✅ Guardado: meta-groups.csv (602 filas x 7 columnas)
--------------------------------------------------
✅ Guardado: meta-members.csv (24591 filas x 6 columnas)
--------------------------------------------------
✅ Guardado: rsvps.csv (126813 filas x 3 columnas)
--------------------------------------------------


In [6]:
# Configuración visual
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

# ── Carga ──────────────────────────────────────────────────────────────────
meta_members = pd.read_csv('../data/raw/meta-members.csv')
meta_groups = pd.read_csv('../data/raw/meta-groups.csv')
meta_events = pd.read_csv('../data/raw/meta-events.csv')

dfs = {
    "meta_events":  meta_events,
    "meta_groups":  meta_groups,
    "meta_members": meta_members,
}

# ── Inspección básica ──────────────────────────────────────────────────────
for name, df in dfs.items():
    print(f"{'='*50}")
    print(f"📄 {name}")
    print(f"  Shape      : {df.shape}")
    print(f"  Columnas   : {df.columns.tolist()}")
    print(f"  Tipos      :\n{df.dtypes.to_string()}")
    print(f"\n  Primeras filas:")
    display(df.head(3))

# ── Nulos y duplicados ─────────────────────────────────────────────────────
for name, df in dfs.items():
    print(f"\n📄 {name}")
    nulls = df.isnull().sum()
    nulls_pct = (nulls / len(df) * 100).round(2)
    null_df = pd.DataFrame({"nulos": nulls, "% nulos": nulls_pct})
    print(null_df[null_df["nulos"] > 0] if null_df["nulos"].sum() > 0 else "  ✅ Sin nulos")
    dups = df.duplicated().sum()
    print(f"  Duplicados : {dups}")

📄 meta_events
  Shape      : (19307, 4)
  Columnas   : ['event_id', 'group_id', 'name', 'time']
  Tipos      :
event_id      str
group_id    int64
name          str
time          str

  Primeras filas:


,event_id,group_id,name,time
0,243930425,26140018,2017 Nashville Walk to End Alzheimers - Octob...,2017-10-14 12:00:00
1,244208851,25604533,Steak Dinner on the Patio,2017-10-15 00:15:00
2,pxlktnywnbfb,25973656,Schedule Meetup,2017-10-03 23:30:00


📄 meta_groups
  Shape      : (602, 7)
  Columnas   : ['group_id', 'group_name', 'num_members', 'category_id', 'category_name', 'organizer_id', 'group_urlname']
  Tipos      :
group_id         int64
group_name         str
num_members      int64
category_id      int64
category_name      str
organizer_id     int64
group_urlname      str

  Primeras filas:


,group_id,group_name,num_members,category_id,category_name,organizer_id,group_urlname
0,339011,Nashville Hiking Meetup,15838,23,Outdoors & Adventure,4353803,nashville-hiking
1,19728145,Stepping Out Social Dance Meetup,1778,5,Dancing,118484462,steppingoutsocialdance
2,6335372,Nashville soccer,2869,32,Sports & Recreation,108448302,Nashville-soccer


📄 meta_members
  Shape      : (24591, 7)
  Columnas   : ['member_id', 'name', 'hometown', 'city', 'state', 'lat', 'lon']
  Tipos      :
member_id      int64
name             str
hometown         str
city             str
state            str
lat          float64
lon          float64

  Primeras filas:


,member_id,name,hometown,city,state,lat,lon
0,2069,Wesley Duffee-Braun,Brentwood,Brentwood,TN,36.00,-86.79
1,8386,Tim,Nashville,Nashville,TN,36.07,-86.78
2,9205,Brenda,Brentwood,Brentwood,TN,36.00,-86.79



📄 meta_events
  ✅ Sin nulos
  Duplicados : 0

📄 meta_groups
  ✅ Sin nulos
  Duplicados : 0

📄 meta_members
          nulos  % nulos
hometown  19664    79.96
state        94     0.38
  Duplicados : 0
